# Memoria del proyecto

## ¿Incide el turismo (receptivo) en el precio de la vivienda?

![Header_EDA](src/images/Header_EDA.jpg)

**Esta memoria es un libro de instrucciones para poder entender el proyecto**: cuenta qué se ha hecho, en qué orden y **dónde está cada cosa**. Todo está interenlazado para que se pueda profundizar en cada apartado, en caso de que sea necesario. Estos  enlaces llevan al notebook, al dato o a la imagen correspondiente. 



### Mapa del proyecto (de los datos brutos a las conclusiones sobre las hipótesis)

| Paso | Qué se hace | Dónde está |
|---|---|---|
| 1 | Datos brutos de 4 fuentes + tabla de equivalencias de países | ruta: src/data/Brutos/ |
| 2 | Limpieza de cada dataset (un notebook por dataset) | ruta: src/data/notebooks/ |
| 3 | Datasets limpios, con las mismas claves (`iso3` y `anio`) | ruta: src/data/Depurados/ |
| 4 | Unión (merge) de los cuatro datasets y cálculo de medianas | [limpieza_dataset_merge_final](src/notebooks/limpieza_dataset_merge_final.ipynb) |
| 5 | Datasets completos ya unidos | [merge_todos_paises](src/data/Mergeados/merge_todos_paises.csv) |
| 6 | Un notebook por gráfico, y sus imágenes | ruta: src/Visualizaciones |
| 7 | Funciones propias que leen datos y guardan resultados | [archivos.py](src/utils/archivos.py) |
| 8 | Esta memoria | ruta: MEMORIA_EDA.ipynb |

## 1. Introducción

La curisosidad para profundizar en este tema surge a raíz de la Huelga General del pasado 11 de noviembre de 2026 denunciaba un encarecimiento del precio de la vivienda y de los alquileres inasumible para muchos españoles. Cada vez es más común escuchar términos como gentrificación, turistificación, zonas tensionadas o tematización urbana y el encarecimiento del m2 ha obligado a muchos vecinos a vivir cada vez más alejado del centro de las ciudades. Este fenómeno no es solo español y según la prensa especializada, ocurre en otros países del mundo. Por eso, este EDA se formula la siguiente pregunta: ¿Incide el turismo en el precio de la vivienda?


### 1.1 Hipótesis principal (H1)

**¿Los países donde más crece el turismo internacional son también los países donde más sube el precio de la vivienda?**

La lógica es que más turistas aumentan la demanda de alojamiento, no solo a nivel hotelero, sino también extrahotelero (con el impulso de vivienda vacacional, alquileres de temporada, colivings, segunda residencia, multipropiedad, home sharing...). Este aumento de la demanda, podría presionar los precios al alza. 

Este trabajo no pretende demostrar que el turismo cause la subida de la vivienda. Pretende comprobar si ambas variables **se mueven juntas**: en caso de ser así, merecería la pena poner la lupa y analizar todo de manera más detallada.

### 1.2 Hipótesis secundaria (H2)

¿La línea de la inflación acompaña a la línea de la subida de los precios de la vivienda? Para hacer el análisis más completo, se ha introducido la variable inflación. Cuanto más se encarece el valor del dinero, mayor es el precio de la vivienda. De esta forma, los gráficos revelarán un lienzo más completo. 

### 1.3 Variable de contexto (H3) 

¿Acompaña el crecimiento de la economía de cada país (PIB per cápita) al crecimiento del precio de la vivienda? Se incorpora como tercera variable de contexto, para comprobar si el precio de la vivienda se mueve con la riqueza de un país y no solo con el turismo o la inflación.

### 1.4 Cómo se leerán los resultados

Todo el análisis habla de **asociación, no de causalidad**: si dos variables se mueven juntas, no se puede concluir que una provoque la otra. Pir este motivo, en las conclusiones se hablará de indicios

## 2. Fuentes de datos

| Dato | Fuente | Indicador usado | Archivo bruto |
|---|---|---|---|
| Turismo | Organización Mundial del Turismo (UN Tourism), estadísticas de turismo receptivo (*inbound*). [Enlace](https://www.untourism.int/es/la-omt-recopila-sistematicamente-estadisticas-turisticas) | `INBD_TRIP_TOTL_TOTL_TOUR`: llegadas de turistas extranjeros que pernoctan, en miles | [2_OMT_INBOUND TOURISM.xlsx](<src/data/Brutos/2_OMT_INBOUND%20TOURISM.xlsx>) |
| Precio de la vivienda | Bank for International Settlements (BIS). Se ha utilizado el dataset publicado en Datahub, elaborado a partir de datos del BIS. [BIS](https://data.bis.org/topics/RPP) · [Datahub](https://datahub.io/core/house-prices-global) | Variación interanual del precio de la vivienda (%), trimestral | [1_BIS DATA_DATA HUB_PRECIO VIVIENDA.csv](<src/data/Brutos/1_BIS DATA_DATA HUB_PRECIO VIVIENDA.csv>) |
| Inflación | Banco Mundial. [Indicador FP.CPI.TOTL.ZG](https://data.worldbank.org/indicator/FP.CPI.TOTL.ZG) | Inflación anual de precios al consumidor (%) | [3_BANCO MUNDIAL_INFLACION POR PAISES.csv](<src/data/Brutos/3_BANCO MUNDIAL_INFLACION POR PAISES.csv>) |
| PIB per cápita | Banco Mundial. [Indicador NY.GDP.PCAP.CD](https://data.worldbank.org/indicator/NY.GDP.PCAP.CD) | PIB per cápita en dólares corrientes | [5_BANCO MUNDIAL_PIB PER CAPITA.csv](<src/data/Brutos/5_BANCO MUNDIAL_PIB PER CAPITA.csv>) |
| Códigos de país | Tabla de equivalencias (códigos ONU, ISO2 e ISO3, según ISO 3166-1) | Sirve para dar a todos los países el mismo código `iso3` | [4_EQUIVALENCIAS_PAISES_ISO_3.csv](src/data/Brutos/4_EQUIVALENCIAS_PAISES_ISO_3.csv) |

**¿Por qué hace falta la tabla de equivalencias):** porque cada fuente identifica los países de una forma distinta (la OMT usa el código numérico de la ONU, el BIS usa el código de 2 letras, el Banco Mundial ya trae el de 3). Pasar todos a `iso3` es lo que permite unir los datasets después.

## 3. Limpieza de los datasets

Cada dataset se ha limpiado en su **propio notebook**. He seguido reglas de limpieza similares enn cuanto a orden, nomenclatura... , para que luego encajen al mergearlos:

- **Clave común:** `iso3` (país) y `anio` (año, como número entero).
- **Periodo analizado:** 2015–2024. Es el tramo en el que hay datos en todas las fuentes (el turismo no tiene datos de 2025).
- **Nombres homogéneos** de las columnas: `pais`, `iso3`, `anio`, y la variable en forma de tasa de crecimiento anual en %.
- **Solo países reales:** se quitan los agregados (Mundo, Unión Europea, grupos de renta...) y los territorios sin código `iso3`.
- Cada notebook acaba **guardando un CSV limpio** en `data/Depurados/`.

### 3.1 Limpieza dataset Vivienda (BIS). El dataset es nominal (no tiene en cuenta la inflación)

Notebook: [limpieza_vivienda.ipynb](src/notebooks/limpieza_vivienda.ipynb) → Resultado: [limpio_vivienda.csv](src/data/Depurados/limpio_vivienda.csv)

| Paso | Qué se hace | Filas × columnas |
|---|---|---|
| 1 | Se parte del dataset bruto (trimestral, con fecha y país) | 8.827 × 4 |
| 2 | Se añade el código `iso3` cruzando el código de 2 letras con la tabla de equivalencias | |
| 3 | Se renombran las columnas y se convierte el año a número (primeros 4 caracteres de la fecha) | |
| 4 | Se acota el periodo a 2015–2024 | 2.440 × 4 |
| 5 | Se eliminan las filas sin `iso3` (zonas agregadas) | 2.280 × 4 |
| 6 | Se calcula la **media anual** de los trimestres de cada país | **570 × 4** |

**Resultado:** 57 países, 10 años cada uno (2015–2024), con la variable `crec._precio_(%)`.

### 3.2 Turismo (OMT)

Notebook: [limpieza_turismo.ipynb](src/notebooks/limpieza_turismo.ipynb) → Resultado: [limpio_turistas_pernoctan.csv](src/data/Depurados/limpio_turistas_pernoctan.csv)

| Paso | Qué se hace | Filas × columnas |
|---|---|---|
| 1 | Se parte del Excel bruto y se trabaja el Excel gracias al import de la librería correspondiente | 13.900 × 13 |
| 2 | Se filtra el indicador de turistas extranjeros que pernoctan | 5.241 × 13 |
| 3 | Se eliminan las columnas que no interesan y se pasa de miles a unidades (× 1.000) para que sea más comprensible al lector | 5.241 × 4 |
| 4 | Se añade `iso3` con la tabla de equivalencias (código de la ONU) para posterior mergeo | 5.241 × 5 |
| 5 | Se eliminan los países sin `iso3` | 5.193 |
| 6 | Se ordena y se acota a 2015-2024 al no disponer data completa del año 2025 | 1.518 × 4 |
| 7 | Se calcula el **crecimiento anual en %** de cada país: año *n* frente a año *n-1* | **1.518 × 5** |

**Resultado:** 194 países. El primer año de cada país (2015) no tiene crecimiento, porque no hay 2014 con el que compararlo: quedan 194 huecos. Esos NaN se eliminan posteriormente al unir los datasets.

### 3.3 Inflación (Banco Mundial)

Notebook: [limpieza_inflacion.ipynb](src/notebooks/limpieza_inflacion.ipynb) → Resultado: [limpio_inflacion.csv](src/data/Depurados/limpio_inflacion.csv)

| Paso | Qué se hace | Filas × columnas |
|---|---|---|
| 1 | Se parte del CSV bruto (un año por columna, de 1960 a 2025) | 266 × 71 |
| 2 | Se seleccionan país, código y los años 2015-2024 | |
| 3 | Se quitan los agregados (Mundo, UE, grupos de renta...) cruzando con la tabla de equivalencias | 215 × 12 |
| 4 | Se pasa de formato ancho (año en columnas) a largo (una fila por país y año) con `melt` para preparar el mergeo posterior | 2.150 × 4 |
| 5 | Se convierte el año a entero y se eliminan los huecos | **1.811 × 4** |

**Resultado:** 189 países. La inflación ya viene como tasa anual en %, así que no hace falta calcular nada más.

### 3.4 PIB per cápita (Banco Mundial)

Notebook: [limpieza_pib.ipynb](src/notebooks/limpieza_pib.ipynb) → Resultado: [limpio_pib.csv](src/data/Depurados/limpio_pib.csv)

| Paso | Qué se hace | Filas × columnas |
|---|---|---|
| 1 | Se parte del CSV bruto | 266 × 71 |
| 2 | Se seleccionan país, código y los años 2015-2024 | |
| 3 | Se quitan los agregados que generan ruido (agrupaciones por continentes, agrupaciones de países) | 215 |
| 4 | Se pasa a formato largo con `melt` y se convierte el año a entero | 2.150 × 4 |
| 5 | Se calcula el **crecimiento anual en %** del PIB per cápita de cada país | |
| 6 | Se eliminan los huecos (el primer año de cada país no tiene crecimiento) | **1.858 × 5** |

**Resultado:** 209 países, periodo 2016-2024.

## 4. Desarrollo del análisis

Notebook de trabajo: [limpieza_dataset_merge_final.ipynb](src/notebooks/limpieza_dataset_merge_final.ipynb)

### 4.1 Unión de los cuatro datasets (merge)

Se parte de la tabla de **vivienda** (570 filas), porque es la variable que se quiere explicar, y se le añaden por la izquierda (`how="left"`) el turismo, la inflación y el PIB, usando como clave `iso3` + `anio`. Así no se pierde ninguna fila de vivienda aunque falte algún dato en las otras.

| Paso | Resultado |
|---|---|
| Vivienda + turismo + inflación + PIB | 570 filas |
| Huecos detectados | Se comprueban con `isnull()`: faltan sobre todo datos de turismo y de PIB (añadir recuento) |
| Se eliminan las filas con algún hueco | **408 filas, 54 países** |

### 4.2 Cuántos países hay con la serie completa

Eliminar huecos quita **filas**, no países: un país puede quedarse con algunos años y perder otros. Para saber cuántos países tienen todos los años (2016-2024) se cuenta cuántas filas tiene cada uno:

| Años con datos | Nº de países | Acumulado |
|---|---|---|
| 9 (serie completa) | **25** | 25 |
| 7 | 18 | 43 |
| 6 | 6 | 49 |
| 5 | 1 | 50 |
| 4 | 4 | 54 |
| **Total** | **54** | |

### 4.3 Qué muestra se usa y por qué

- **Serie mundial por año (gráficos de evolución):** se utiliza el **panel de 25 países** con los 9 años completos. Así, cada año se calcula con **los mismos países** y la evolución es comparable. Se prioriza una serie completa y comparable (incluidos los años posteriores a la pandemia) frente a una muestra más grande pero con años distintos por país.
- **Alternativa valorada:** limitar el periodo a 2016-2022 permitiría usar 42 países, pero obligaría a descartar 2023 y 2024, que son años que nos interesan al estar fuera del rango 2020-2022 de la pandemia.
- **Correlaciones:** se utilizan las **408 observaciones** de los 54 países (más datos), y se repiten con el panel de 25 y sin los años 2020-2022 como **comprobación de robustez**.

### 4.4 Por qué mediana y no media

Para resumir "el país típico" de cada año se usa la **mediana**. La pandemia de COVID-19 genera valores extremos (*outliers*), sobre todo en el crecimiento del turismo (caída en 2020 y rebote en 2021-2022 desde una base muy baja), y la media se deja arrastrar por ellos. Un ejemplo de las pruebas iniciales: con la media de un panel de 42 países, el crecimiento del turismo en 2022 salía un **433,5 %**; con la mediana del panel de 25 países sale un **136,2 %**. La mediana es más robusta a esos extremos.

Mediana mundial por año (panel de 25 países), guardada en [merge_mediana_mundial.csv](src/data/Mergeados/merge_mediana_mundial.csv):

| Año | Precio vivienda (%) | Turistas (%) | Inflación (%) | PIB per cápita (%) |
|---|---|---|---|---|
| 2016 | 5,4 | 9,3 | 0,6 | 2,8 |
| 2017 | 5,3 | 9,9 | 2,0 | 7,4 |
| 2018 | 5,8 | 5,1 | 2,0 | 7,4 |
| 2019 | 3,7 | 2,5 | 1,7 | -0,4 |
| 2020 | 4,6 | -74,4 | 1,0 | -3,8 |
| 2021 | 8,2 | 6,7 | 3,1 | 16,1 |
| 2022 | 8,9 | 136,2 | 8,0 | -0,4 |
| 2023 | 4,0 | 21,1 | 5,5 | 10,2 |
| 2024 | 6,0 | 8,6 | 2,9 | 4,9 |

Mediana de toda la serie (2016-2024): precio de la vivienda **6,0 %**, turistas **7,9 %**, inflación **2,4 %**, PIB per cápita **4,4 %**.

### 4.5 Revalorización real de la vivienda

El precio de la vivienda está en términos **nominales** (sin descontar la inflación). Para saber si una vivienda se revaloriza de verdad, se calcula el **crecimiento real** de forma aproximada: `crecimiento real = crecimiento del precio - inflación`. Esta resta es una aproximación válida con inflaciones bajas y moderadas; con inflaciones muy altas (por ejemplo, Türkiye) exagera el resultado.

Datos: [merge_vivienda_inflacion.csv](src/data/Mergeados/merge_vivienda_inflacion.csv)

### 4.6 Datasets completos guardados

Para no repetir cálculos, los datasets unidos se guardan en la carpeta `src/data/Mergeados` y cada notebook de gráficos los lee desde ahí:

| Archivo | Contenido |
|---|---|
| [merge_todos_paises.csv](src/data/Mergeados/merge_todos_paises.csv) | 54 países, 408 observaciones |
| [merge_panel_25_paises_16-24.csv](src/data/Mergeados/merge_panel_25_paises_16-24.csv) | 25 países con serie completa, 225 observaciones |
| [merge_mediana_mundial.csv](src/data/Mergeados/merge_mediana_mundial.csv) | Mediana por año del panel de 25 países |
| [merge_vivienda_inflacion.csv](src/data/Mergeados/merge_vivienda_inflacion.csv) | Vivienda + inflación, con el crecimiento real |

### 4.7 Funciones propias

[archivos.py](src/utils/archivos.py) reúne las funciones que usan varios notebooks:

| Función | Qué hace |
|---|---|
| `cargar_dataset(nombre, carpeta="Mergeados")` | Lee un CSV de `data/<carpeta>/` |
| `guardar_dataset(df, nombre, carpeta="Mergeados", index=False)` | Guarda una tabla como CSV |
| `guardar_grafico(nombre)` | Guarda el gráfico actual como PNG en `visualizaciones/` | CSV |
| `guardar_grafico(nombre)` | Guarda el gráfico actual como PNG en `visualizaciones/` |

## 5. Visualizaciones

Cada gráfico tiene su propio notebook en la carpeta `src/Visualizaciones`. Debajo de cada imagen va una descripción de lo que se ve.

### Gráfico 1. Evolución de la mediana mundial (vista completa)
Notebook: [01_evolucion_mediana_mundial_1.ipynb](src/Visualizaciones/01_evolucion_mediana_mundial_1.ipynb)

![Gráfico 1](src/Visualizaciones/01_PNG_evolucion_mediana_mundial.png)

Las cuatro variables, 2016-2024. El turismo domina la escala por el hundimiento de 2020 (-74,4 %) y el rebote de 2022 (+136,2 %). Las demás se mueven en un rango mucho menor.

### Gráfico 2. La misma evolución, con zoom
Notebook: [02_evolucion_mediana_mundial_zoom_1.ipynb](src/Visualizaciones/02_evolucion_mediana_mundial_zoom_1.ipynb)

![Gráfico 2](src/Visualizaciones/02_PNG_evolucion_mediana_mundial_zoom.png)

El eje vertical se limita de -5 a 20 para ver mejor cómo se mueven precio de la vivienda, PIB e inflación. El turismo se sale del marco en 2020 y 2022 (ver el gráfico 1).

### Gráfico 3. Top 15 de revalorización de la vivienda (nominal)
Notebook: [03_top15_revalorizacion_nominal.ipynb](src/Visualizaciones/03_top15_revalorizacion_nominal.ipynb)

![Gráfico 3](src/Visualizaciones/03_PNG_top15_revalorizacion_nominal.png)

Ranking de crecimiento del precio de la vivienda (indicar el periodo/medida exacta que usa el notebook). Lideran Türkiye (41,0 %), Hungría (13,5 %) e Islandia (10,6 %). España queda en el puesto 30 de 57 (5,2 %).

### Gráfico 4. Top 15 de revalorización real (descontada la inflación)
Notebook: [04_top15_revalorizacion_real.ipynb](src/Visualizaciones/04_top15_revalorizacion_real.ipynb)

![Gráfico 4](src/Visualizaciones/04_PNG_top15_revalorizacion_real.png)

Al descontar la inflación, el ranking cambia: Türkiye baja a 13,6 %, y entran países como Nueva Zelanda, Croacia e Israel. España queda en el puesto 26 de 57 (3,1 %).

### Gráfico 5. Mapa de calor de correlaciones
Notebook: [05_correlaciones.ipynb](src/Visualizaciones/05_correlaciones.ipynb)

![Gráfico 5](src/Visualizaciones/05_PNG_correlaciones.png)

Correlación entre las cuatro variables (54 países, 408 observaciones). La fila del precio de la vivienda responde a las hipótesis: turismo -0,02 (H1), inflación 0,73 (H2) y PIB 0,17 (H3). El mismo notebook incluye la comprobación de robustez.

### Gráfico 6. Precio de la vivienda frente a inflación (H2)
Notebook: [06_vivienda_vs_inflacion.ipynb](src/Visualizaciones/06_vivienda_vs_inflacion.ipynb)

![Gráfico 6](src/Visualizaciones/06_PNG_vivienda_vs_inflacion.png)

Las dos líneas suben a la vez entre 2019 y 2022 (la inflación pasa de 1,7 % a 8,0 % y el precio de 3,7 % a 8,9 %). En 2023 ambas bajan, y en 2024 se separan. El precio crece por encima de la inflación todos los años salvo 2023.

## 6. Conclusiones 

### 6.1 Sobre las hipótesis

| Hipótesis | Qué se observa | Lectura |
|---|---|---|
| **H1.** Turismo y precio de la vivienda | Los datos están desvirtuados por el cierre de fronteras durante la crisis del coronavirus. Correlación de −0,02 con las 408 observaciones. En la comprobación de robustez (sin 2020–2022 y con el panel de 25) los valores también son bajos, cercanos a 0 y de signo ligeramente negativo *(a confirmar al ejecutar el notebook 05)* | **No hay indicios de que ambas variables se muevan juntas en estos datos** |
| **H2.** Inflación y precio de la vivienda | Correlación de 0,73 con todas las observaciones, pero baja a valores moderados o bajos al quitar 2020–2022 o usar el panel. En el gráfico 6 las líneas suben juntas entre 2019 y 2022 | Hay indicios de asociación positiva, pero **poco estable**: depende de los años y países con inflaciones extremas |
| **H3.** PIB y precio de la vivienda | Correlación de 0,17 con todas las observaciones; valores parecidos y bajos en las comprobaciones| Asociación positiva débil |

### 6.2 Sobre el precio de la vivienda en el mundo

- La mediana del crecimiento anual del precio de la vivienda ronda el **6 %**, por encima de la inflación mediana (**2,4 %**): en conjunto, la vivienda se ha revalorizado también en términos reales, por lo que su evolución apuntala su percepción de valor refugio como inversión.
-   Eso sí, hay mercados donde el precio de la vivienda pierde valor real. Al descontar la inflación, ejemplos como Perú, Brasil o Marruecos quedan en negativo.
-   España ocupa el puesto 30 de 57 en revalorización nominal y el 26 en real: una posición intermedia, no extrema.

### 6.3 Limitaciones

- **Pocos años:** la serie mundial tiene 9 puntos (2016–2024), y la pandemia ocupa buena parte de ellos, contaminando los datos del dataset de la OMT para extraer conclusiones más sólidas.
- **Muestra:** solo 25 países tienen la serie completa. Se parte de 57 países de vivienda (BIS), que son sobre todo economías avanzadas y algunas emergentes, no el mundo entero.
- **Turismo:** el crecimiento tras 2020 es un efecto de base (se rebota desde una caída enorme), no un crecimiento "normal".
- **PIB:** se mide en dólares corrientes, así que **mezcla crecimiento económico con variaciones del tipo de cambio**.
- **Vivienda:** son precios nominales; el crecimiento real es una aproximación (precio − inflación).
- **Asociación no es causalidad:** correlaciones bajas o altas no prueban que una variable provoque la otra.

### 6.4 Oportunidades y siguientes pasos

- Analizar el caso de ciudades concretas (el turismo suele concentrarse en pocas), en vez de países enteros.
- Ampliar el análisis con el alquiler y con datos de vivienda vacacional.
- Incluir más variables: aumento de la población, evolución de la oferta de vivienda, evolución de los tipos de interés (acceso a hipotecas), .
- Incluir más países y más años cuando haya datos de turismo completos.

## 7. Cómo reproducir el proyecto

Orden recomendado para ejecutar todo desde cero (Restart + Run All en cada notebook):

1. Los cuatro notebooks de limpieza de [`notebooks/`](notebooks/): vivienda, turismo, inflación y PIB. Generan los CSV de `data/Depurados/`.
2. [`notebooks/limpieza_dataset_merge_final.ipynb`](notebooks/limpieza_dataset_merge_final.ipynb): une los datasets y guarda los CSV de `data/Mergeados/`.
3. Los seis notebooks de [`visualizaciones/`](visualizaciones/), en orden del 01 al 06. Generan las imágenes.

*Pendiente:* un `main.py` que ejecute esta cadena entera con funciones.

## 8. Uso de IA 

Durante el proyecto me he apoyado en Claude en la revisión del código, y apoyo en dudas, preguntas y otras cuestiones. Se ha usado de manera responsable responsable y siempre enfocada a aportar mayor calidad y mejorar el aprendizaje